# Januário Leads — Fase 7 · Recuperação FINAL dos municípios pendentes

**Novo Colab independente, criado especificamente para os municípios restantes.** Ele reaproveita o mesmo importador confiável da Fase 6, mas usa a **malha oficial do IBGE** para reconciliar os registros de recuperação com municípios realmente ausentes.

**Estado de referência (13/11/2026):** 5.552/5.570 localidades no banco de HOMOLOGAÇÃO, 288.734 empresas, aproximadamente 163 MiB. Havia 22 entradas de recuperação para uma diferença numérica de 18 municípios; elas **não são necessariamente 22 cidades oficiais distintas ainda ausentes**. Por isso o notebook não usa uma lista fixa de 18 nomes: verifica o banco e a malha IBGE antes de qualquer escrita.

### Como usar
1. No Colab, execute as células **1, 2 e 3** (instalação, importador e configurações).
2. Execute a **célula 4** para diagnosticar e recuperar os municípios faltantes. Digite `RECUPERAR_18` quando solicitado.
3. Cole a **chave secreta do projeto de HOMOLOGAÇÃO** no campo oculto. **Nunca cole a chave no chat, no notebook ou no GitHub**.
4. Veja o resultado `RESUMO FINAL`. Se ainda restarem cidades com `retry_after` futuro, rode **somente a célula 4** depois da data indicada.

### Proteções
- Não acessa a produção. Não modifica os notebooks das fases anteriores.
- **Não exclui** empresas ou municípios e não reinicia a base. Confere o progresso antes e depois de cada tentativa.
- Limite de **60 empresas por município** (amostra, não cobertura integral), parada preventiva em **340 MiB**, máximo de **350 MiB** na execução do importador.
- Respeita o período de recuperação e faz no máximo duas tentativas por cidade para falhas transitórias.
- Se houver divergência entre os nomes e os totais, informa o problema; **não inventa municípios concluídos**.
- O token só vive na execução da célula 4 e é enviado oculto ao processo de importação; desconecte o runtime ao terminar.

> Observação: municípios sem lugares utilizáveis na fonte Overture **podem continuar pendentes**. O notebook nunca grava registros fictícios para atingir artificialmente 100%.

**Atualização de segurança (13/11/2026):** teto interno do importador Overture ajustado para **350 MiB**, com pausa em **340 MiB**. A cota real do Supabase Free permanece 500 MB por projeto. A alteração não modifica o limite de 60 estabelecimentos/cidade, não inicia importações e não garante +40 empresas em todos os municípios.


In [ ]:
# Célula 1 — instalar ferramentas necessárias
%pip install -q "overturemaps>=2.1,<3" "geopandas>=1,<2" "shapely>=2,<3" "pyogrio>=0.10,<1"
print("Dependências instaladas.")

In [ ]:
# Célula 2 — disponibilizar o importador localmente, sem acessar o GitHub privado.
# Cópia do script do Januário Leads; não contém chave ou segredo.
from pathlib import Path
_IMPORTER = (
    "#!/usr/bin/env python3\n\"\"\"Import Overture Places into private Supabase table, scoped to Brazil's municipalitie"
    "s.\n\nPre-req:\n    pip install overturemaps geopandas shapely pyogrio\n    export SUPABASE_URL='https://<project>"
    ".supabase.co'\n    export SUPABASE_SERVICE_ROLE_KEY='<service role secret>'\n    python scripts/import_overture_"
    "br.py --states MG,SP --database-max-mb 400\n\nThe runner covers all 27 federal units by default. Each state is s"
    "treamed\nseparately; it does not attempt to download and retain a global Overture file.\nIt stops BEFORE reachin"
    "g a configured database threshold, and reports partial\ncoverage honestly. Never run from a browser or publish "
    "the service-role key.\n\"\"\"\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subp"
    "rocess\nimport sys\nimport tempfile\nimport time\nimport unicodedata\nfrom urllib import request, error\nfrom urllib"
    ".parse import urlparse\n\nIBGE_2024_ROOT = (\n    \"https://geoftp.ibge.gov.br/organizacao_do_territorio/\"\n    \"ma"
    "lhas_territoriais/malhas_municipais/municipio_2024\"\n)\n\ndef state_boundaries_url(state):\n    \"\"\"IBGE offers one"
    " ZIP per UF; do not download the 199 MB national file.\"\"\"\n    if state not in STATE_CODES:\n        raise Value"
    "Error(\"Invalid Brazilian state code\")\n    return f\"{IBGE_2024_ROOT}/UFs/{state}/{state}_Municipios_2024.zip\"\n\n"
    "def ensure_state_boundaries(state, source=None):\n    if source:\n        path = Path(source)\n        if not pat"
    "h.is_file():\n            raise RuntimeError(f\"IBGE boundaries file not found: {path}\")\n        return path\n   "
    " path = Path(\"data/ibge/\") / f\"{state}_Municipios_2024.zip\"\n    if not path.is_file():\n        path.parent.mkd"
    "ir(parents=True, exist_ok=True)\n        url = state_boundaries_url(state)\n        print(f\"[{state}] Downloadin"
    "g IBGE municipal boundaries: {url}\", flush=True)\n        temp = path.with_suffix(\".partial\")\n        try:\n    "
    "        request.urlretrieve(url, temp)\n            if temp.stat().st_size < 512:\n                raise Runtime"
    "Error(\"IBGE boundaries download appears empty\")\n            temp.rename(path)\n        finally:\n            tem"
    "p.unlink(missing_ok=True)\n    return path\nSTATE_CODES = (\n    \"AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR"
    " PE PI \"\n    \"RJ RN RS RO RR SC SP SE TO\"\n).split()\nBATCH_SIZE = 150\n\n\ndef normalize(value):\n    \"\"\"Match the "
    "TypeScript search adapter's city/name normalization exactly.\"\"\"\n    letters = \"\".join(\n        c for c in unic"
    "odedata.normalize(\"NFD\", value.casefold())\n        if unicodedata.category(c) != \"Mn\"\n    )\n    return \" \".joi"
    "n(re.sub(r\"[^a-z0-9]+\", \" \", letters).split())\n\n\ndef first(values):\n    return next((v for v in (values or [])"
    " if isinstance(v, str) and v.strip()), None)\n\n\n# A social profile, WhatsApp contact or link shortener is NOT a"
    " verified\n# business website. This preserves truthful \"sem site\" prospecting filters.\nNON_WEBSITE_HOSTS = froz"
    "enset((\n    \"wa.me\", \"whatsapp.com\", \"api.whatsapp.com\", \"instagram.com\",\n    \"facebook.com\", \"fb.me\", \"m.me\","
    " \"linktr.ee\", \"linktree.com\",\n    \"bit.ly\", \"tinyurl.com\", \"t.co\", \"youtube.com\", \"youtu.be\",\n    \"tiktok.com\""
    ", \"x.com\", \"twitter.com\", \"maps.google.com\",\n    \"goo.gl\", \"beacons.ai\",\n))\n\n\ndef is_business_website(value):\n"
    "    if not isinstance(value, str):\n        return False\n    try:\n        url = urlparse(value.strip())\n       "
    " host = (url.hostname or \"\").lower().removeprefix(\"www.\")\n        return (url.scheme in (\"http\", \"https\")\n    "
    "            and bool(host) and \".\" in host\n                and host not in NON_WEBSITE_HOSTS\n                a"
    "nd not any(host.endswith(\".\" + blocked)\n                            for blocked in NON_WEBSITE_HOSTS))\n    exc"
    "ept ValueError:\n        return False\n\n\ndef first_website(values):\n    return first([value for value in (values"
    " or []) if is_business_website(value)])\n\n\ndef select_diverse_pool(records, maximum):\n    \"\"\"Reserve 40% for co"
    "ntactable companies without identified websites.\n\n    Fill unused slots with the best available business conta"
    "cts. Never promise\n    that a missing website is proof a business actually has no website.\n    \"\"\"\n    ranked "
    "= sorted(records, key=lead_quality, reverse=True)\n    no_site = [r for r in ranked if not r.get(\"website\")\n   "
    "            and (r.get(\"phone\") or r.get(\"email\") or r.get(\"instagram\"))]\n    target_no_site = min(len(no_site"
    "), (maximum * 2 + 4) // 5)\n    chosen = no_site[:target_no_site]\n    selected = {r[\"id\"] for r in chosen}\n    "
    "for row in ranked:\n        if len(chosen) >= maximum:\n            break\n        if row[\"id\"] not in selected:\n"
    "            chosen.append(row)\n            selected.add(row[\"id\"])\n    return chosen\n\n\ndef api(base, key, path"
    ", method=\"GET\", payload=None):\n    data = json.dumps(payload, ensure_ascii=False).encode() if payload is not N"
    "one else None\n    # Legacy service_role JWTs use Authorization; the newer sb_secret_ keys\n    # are opaque and"
    " must be sent only as apikey, never as Bearer JWT.\n    if not (key.startswith(\"eyJ\") or key.startswith(\"sb_sec"
    "ret_\")):\n        raise RuntimeError(\"Use somente a chave secreta de STAGING (sb_secret_ ou service_role legada"
    ").\")\n    headers = {\n        \"apikey\": key,\n        \"Content-Type\": \"application/json\",\n        \"Prefer\": \"res"
    "olution=merge-duplicates,return=minimal\",\n    }\n    if key.startswith(\"eyJ\"):\n        headers[\"Authorization\"]"
    " = f\"Bearer {key}\"\n    for attempt in range(5):\n        try:\n            req = request.Request(base + path, me"
    "thod=method, headers=headers, data=data)\n            with request.urlopen(req, timeout=45) as resp:\n          "
    "      body = resp.read()\n                return json.loads(body) if body else None\n        except error.HTTPEr"
    "ror as exc:\n            detail = exc.read(300).decode(\"utf-8\", errors=\"replace\")\n            if exc.code not i"
    "n (429, 500, 502, 503, 504) or attempt == 4:\n                raise RuntimeError(f\"Supabase API {exc.code}: {de"
    "tail}\") from exc\n        except (error.URLError, TimeoutError):\n            if attempt == 4:\n                r"
    "aise\n        time.sleep(min(2 ** attempt, 15))\n    raise RuntimeError(\"Unreachable API retry state\")\n\n\ndef dat"
    "abase_bytes(base, key):\n    result = api(base, key, \"/rest/v1/rpc/overture_storage_bytes\", method=\"POST\", payl"
    "oad={})\n    if not isinstance(result, int):\n        raise RuntimeError(\"Storage guard unavailable: run SQL mig"
    "ration first.\")\n    return result\n\n\ndef status(base, key, state, phase, count=0, message=None):\n    api(base, "
    "key, \"/rest/v1/overture_import_progress?on_conflict=state\", \"POST\", [{\n        \"state\": state, \"status\": phase"
    ",\n        \"imported_rows\": count, \"last_error\": message,\n        \"checked_at\": time.strftime(\"%Y-%m-%dT%H:%M:%"
    "SZ\", time.gmtime()),\n    }])\n\n\ndef municipalities_index(boundaries):\n    import geopandas as gpd\n\n    frame = "
    "gpd.read_file(\"zip://\" + str(boundaries.resolve()))\n    if frame.crs is None:\n        raise RuntimeError(\"The "
    "IBGE municipal boundary file has no CRS.\")\n    frame = frame.to_crs(epsg=4326).reset_index(drop=True)\n    colu"
    "mns = set(frame.columns)\n    if not {\"NM_MUN\", \"SIGLA_UF\"}.issubset(columns):\n        raise RuntimeError(f\"Une"
    "xpected IBGE municipalities columns: {sorted(columns)}\")\n    frame = frame[frame[\"SIGLA_UF\"].isin(STATE_CODES)"
    "].copy().reset_index(drop=True)\n    return frame\n\n\ndef place_record(feature, municipalities, state):\n    from "
    "shapely.geometry import Point\n\n    props = feature.get(\"properties\") or {}\n    coords = (feature.get(\"geometry"
    "\") or {}).get(\"coordinates\") or []\n    if len(coords) != 2:\n        return None\n    lon, lat = coords\n    if n"
    "ot isinstance(lon, (int, float)) or not isinstance(lat, (int, float)):\n        return None\n    name = (props.g"
    "et(\"names\") or {}).get(\"primary\")\n    # GeoJSON exporters may place the Overture GERS id in Feature.id rather\n"
    "    # than properties.id. Preserve either representation.\n    identifier = props.get(\"id\") or feature.get(\"id\""
    ")\n    if not isinstance(name, str) or not name.strip() or not isinstance(identifier, str):\n        return None"
    "\n    if props.get(\"operating_status\") == \"permanently_closed\":\n        return None\n\n    # State bounding boxes"
    " overlap: map point to the true municipality.\n    point = Point(lon, lat)\n    matches = municipalities.sindex."
    "query(point, predicate=\"intersects\")\n    if len(matches) == 0:\n        return None\n    mun = municipalities.il"
    "oc[int(matches[0])]\n    if mun[\"SIGLA_UF\"] != state:\n        return None\n    address = (props.get(\"addresses\")"
    " or [{}])[0]\n    if not isinstance(address, dict):\n        address = {}\n    taxonomy = props.get(\"taxonomy\") o"
    "r {}\n    confidence = props.get(\"confidence\")\n    socials = props.get(\"socials\") or []\n    websites = props.ge"
    "t(\"websites\") or []\n    instagram = first([s for s in (socials + websites)\n                       if isinstanc"
    "e(s, str) and\n                       re.search(r\"https?://(?:www\\.)?instagram\\.com/\", s, re.I)])\n    return {\n"
    "        \"id\": identifier[:100],\n        \"name\": name.strip()[:240],\n        \"name_key\": normalize(name)[:240],"
    "\n        \"category\": str(taxonomy.get(\"primary\") or props.get(\"basic_category\") or \"\")[:120] or None,\n        "
    "\"city\": str(mun[\"NM_MUN\"])[:120],\n        \"city_key\": normalize(str(mun[\"NM_MUN\"]))[:120],\n        \"state\": st"
    "r(mun[\"SIGLA_UF\"]),\n        \"address\": str(address.get(\"freeform\") or \"\")[:400] or None,\n        \"latitude\": l"
    "at, \"longitude\": lon,\n        \"phone\": (first(props.get(\"phones\")) or \"\")[:80] or None,\n        \"email\": (firs"
    "t(props.get(\"emails\")) or \"\")[:200] or None,\n        \"website\": (first_website(websites) or \"\")[:500] or None,"
    "\n        \"instagram\": instagram[:500] if instagram else None,\n        \"confidence\": confidence if isinstance(c"
    "onfidence, (int, float)) else None,\n        \"operating_status\": str(props.get(\"operating_status\") or \"\")[:35] "
    "or None,\n        \"data_release\": os.getenv(\"OVERTURE_DATA_RELEASE\", \"latest\")[:80],\n    }\n\n\ndef lead_quality(r"
    "ow):\n    \"\"\"Prioritize genuine, contactable establishments within the per-city quota.\"\"\"\n    confidence = row."
    "get(\"confidence\")\n    weight = float(confidence) * 5 if isinstance(confidence, (int, float)) else 0\n    return"
    " (weight\n            + 12 * bool(row.get(\"phone\"))\n            + 11 * bool(row.get(\"email\"))\n            + 7 *"
    " bool(row.get(\"website\"))\n            + 3 * bool(row.get(\"instagram\"))\n            + 2 * bool(row.get(\"categor"
    "y\")))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--state"
    "s\", default=\",\".join(STATE_CODES),\n                        help=\"Comma-separated UF codes; default all 27\")\n  "
    "  parser.add_argument(\"--cities\", default=\"\",\n                        help=\"Optional comma-separated cities, r"
    "equires exactly one UF; useful for a small pilot\")\n    parser.add_argument(\"--boundaries\", default=None,\n     "
    "                   help=\"Optional existing ZIP; defaults to one IBGE ZIP per UF\")\n    parser.add_argument(\"--d"
    "atabase-max-mb\", type=int, default=400)\n    parser.add_argument(\"--max-records-per-state\", type=int, default=0"
    ",\n                        help=\"0=no state cap; nonzero limits state rows and marks coverage partial\")\n    par"
    "ser.add_argument(\"--max-records-per-city\", type=int, default=20,\n                        help=\"Per-municipalit"
    "y safety cap, default 20 (0=unlimited)\")\n    parser.add_argument(\"--dry-run\", action=\"store_true\")\n    args = "
    "parser.parse_args()\n    if args.database_max_mb < 30 or args.database_max_mb > 480:\n        parser.error(\"--da"
    "tabase-max-mb must be between 30 and 480 MB\")\n    if args.max_records_per_state < 0:\n        parser.error(\"--m"
    "ax-records-per-state cannot be negative\")\n    if not 1 <= args.max_records_per_city <= 100:\n        parser.err"
    "or(\"--max-records-per-city must be between 1 and 100\")\n    states = [s.strip().upper() for s in args.states.sp"
    "lit(\",\")]\n    if not states or len(set(states)) != len(states) or any(s not in STATE_CODES for s in states):\n "
    "       parser.error(\"--states must contain unique Brazilian UF codes\")\n    cities = [normalize(s.strip()) for "
    "s in args.cities.split(\",\") if s.strip()]\n    if cities and len(states) != 1:\n        parser.error(\"--cities r"
    "equires exactly one UF in --states\")\n\n    base = os.getenv(\"SUPABASE_URL\", \"\").rstrip(\"/\")\n    key = os.getenv"
    "(\"SUPABASE_SERVICE_ROLE_KEY\", \"\")\n    if not args.dry_run and (not base.startswith(\"https://\") or not key):\n  "
    "      parser.error(\"SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY are required\")\n\n    limit_bytes = args.database"
    "_max_mb * 1024 * 1024\n    for state in states:\n        # Individual state archive keeps network and runner-mem"
    "ory use modest.\n        boundaries = ensure_state_boundaries(state, args.boundaries)\n        municipalities = "
    "municipalities_index(boundaries)\n        state_munis = municipalities.loc[municipalities.SIGLA_UF == state].co"
    "py()\n        if cities:\n            state_munis = state_munis.loc[\n                state_munis.NM_MUN.map(lamb"
    "da name: normalize(str(name)) in cities)\n            ].copy()\n            found_cities = {normalize(str(name))"
    " for name in state_munis.NM_MUN}\n            if set(cities) != found_cities:\n                raise RuntimeErro"
    "r(f\"Municipality not found in {state}: {sorted(set(cities) - found_cities)}\")\n        if state_munis.empty:\n  "
    "          raise RuntimeError(f\"IBGE archive does not cover {state}\")\n        west, south, east, north = state_"
    "munis.total_bounds\n        # Slightly wider rectangle avoids border rounding gaps.\n        bbox = \",\".join(f\"{"
    "v:.6f}\" for v in (west - .005, south - .005, east + .005, north + .005))\n        print(f\"[{state}] Overture bb"
    "ox {bbox}\", flush=True)\n        if args.dry_run:\n            continue\n        storage_before = database_bytes("
    "base, key)\n        if storage_before >= limit_bytes:\n            print(f\"STOP: database reached configured {ar"
    "gs.database_max_mb} MB safety limit.\")\n            return 2\n        total = 0\n        status(base, key, state,"
    " \"running\")\n        try:\n            with tempfile.TemporaryDirectory(prefix=\"overture-br-\") as temp:\n        "
    "        out = Path(temp) / f\"{state}.geojsonseq\"\n                subprocess.run([\"overturemaps\", \"download\", f"
    "\"--bbox={bbox}\",\n                                \"--type=place\", \"-f\", \"geojsonseq\", \"-o\", str(out)],\n        "
    "                       check=True)\n                # Store only a small, quality-ranked per-city selection in "
    "memory.\n                city_records = {}\n                city_capped = False\n                with out.open(\"r"
    "\", encoding=\"utf-8\") as src:\n                    for line in src:\n                        line = line.lstrip(\""
    "\\x1e\").strip()\n                        if not line:\n                            continue\n                     "
    "   row = place_record(json.loads(line), state_munis, state)\n                        if row is None:\n          "
    "                  continue\n                        # Keep two bounded quality pools per municipality.\n        "
    "                # Otherwise the top 30 are almost always ALL \"with site\".\n                        groups = cit"
    "y_records.setdefault(row[\"city_key\"], {\n                            \"with_site\": [], \"without_site\": [],\n     "
    "                   })\n                        pool = groups[\"with_site\" if row.get(\"website\") else \"without_si"
    "te\"]\n                        if len(pool) < args.max_records_per_city:\n                            pool.append"
    "(row)\n                        else:\n                            city_capped = True\n                           "
    " worst = min(range(len(pool)),\n                                        key=lambda idx: lead_quality(pool[idx])"
    ")\n                            if lead_quality(row) > lead_quality(pool[worst]):\n                              "
    "  pool[worst] = row\n\n                # Interleave municipality samples for nationwide fair coverage.\n         "
    "       ranked_cities = {\n                    city: select_diverse_pool(groups[\"with_site\"] + groups[\"without_s"
    "ite\"],\n                                              args.max_records_per_city)\n                    for city, "
    "groups in city_records.items()\n                }\n                selected = []\n                for rank in ran"
    "ge(args.max_records_per_city):\n                    for city in sorted(ranked_cities):\n                        "
    "pool = ranked_cities[city]\n                        if rank < len(pool):\n                            selected.a"
    "ppend(pool[rank])\n\n                if not selected:\n                    raise RuntimeError(\"Overture returned "
    "no usable places for this state\")\n                # A municipality-only pilot never marks the whole state as i"
    "mported.\n                partial = city_capped or bool(cities)\n                if args.max_records_per_state a"
    "nd len(selected) > args.max_records_per_state:\n                    selected = selected[:args.max_records_per_s"
    "tate]\n                    partial = True\n                for start in range(0, len(selected), BATCH_SIZE):\n   "
    "                 if database_bytes(base, key) >= limit_bytes:\n                        partial = True\n         "
    "               break\n                    batch = selected[start:start + BATCH_SIZE]\n                    api(ba"
    "se, key, \"/rest/v1/overture_places?on_conflict=id\", \"POST\", batch)\n                    total += len(batch)\n   "
    "             storage_after = database_bytes(base, key)\n                status(base, key, state, \"partial\" if p"
    "artial else \"complete\", total)\n                delta = storage_after - storage_before\n                bytes_pe"
    "r_imported_row = delta // total if total else 0\n                print(f\"[{state}] {total} imported; \"\n        "
    "              f\"database before={storage_before}, after={storage_after}, \"\n                      f\"delta={delt"
    "a}, average={bytes_per_imported_row} B/row; \"\n                      f\"{'PARTIAL' if partial else 'COMPLETE'}\","
    " flush=True)\n                if storage_after >= limit_bytes:\n                    print(\"STOP: database safety"
    " threshold; remaining states NOT imported.\", flush=True)\n                    return 2\n        except Exception"
    " as exc:\n            status(base, key, state, \"error\", total, str(exc)[:200])\n            raise\n    print(\"Ing"
    "estion pass ended. Check overture_import_progress for coverage.\", flush=True)\n    return 0\n\n\nif __name__ == \"_"
    "_main__\":\n    sys.exit(main())\n"
)
_SCRIPT = Path("/content/import_overture_br.py")
compile(_IMPORTER, str(_SCRIPT), "exec")  # valida a sintaxe antes de executar
_SCRIPT.write_text(_IMPORTER, encoding="utf-8")
print("Importador preparado. Compatível com sb_secret_ e service_role legado; nenhum dado enviado ao Supabase.")


In [ ]:
# Célula 3 — configurações da recuperação dirigida (sem conexão ou gravação).
TOTAL_LOCALIDADES_BRASIL = 5570
LIMITE_POR_MUNICIPIO = 60
MAX_NOVOS_POR_SESSAO = 30   # A lista real é calculada; 18 é a diferença inicial, não um conjunto fixo.
TEMPO_MAX_MINUTOS = 240
MAX_FALHAS_POR_EXECUCAO = 9
MAX_FALHAS_CONSECUTIVAS = 5
MAX_UFS_COM_FALHA_DOWNLOAD = 4
ESPERA_ENTRE_TENTATIVAS = 18
ESPERA_ENTRE_CIDADES = 1
HORAS_ESPERA_FALHAS = 6
DIAS_ESPERA_SEM_DADOS = 7
AMPLIAR_MALHA_NACIONAL_SE_DIVERGIR = True
RESPEITAR_AGENDAMENTO_RECUPERACAO = True
# O scanner local nunca chama a API de localidades que causava TimeoutError.
UFS_BRASIL = tuple("AC AL AP AM BA CE DF ES GO MA MG MS MT PA PB PE PI PR RJ RN RO RR RS SC SE SP TO".split())
assert len(UFS_BRASIL)==27 and len(set(UFS_BRASIL))==27
assert 1 <= LIMITE_POR_MUNICIPIO <= 60
assert 1 <= MAX_NOVOS_POR_SESSAO <= 50
assert 15 <= TEMPO_MAX_MINUTOS <= 660
print("FASE 7: só cidades ainda não confirmadas em HOMOLOGAÇÃO.")
print("Fonte de nomes oficiais: malhas municipais IBGE 2024; importar Overture até 60 empresas/cidade.")
print("Célula de configuração pronta. Nenhuma credencial foi solicitada; nenhum dado alterado.")

In [ ]:
# Célula 4 — RETOMADA SEM BARRAS: log curto, tenta novamente, guarda pendências.
# O Supabase é a fonte real para retomar de onde parou.
from getpass import getpass
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError
import json
import os
import re
import subprocess
import sys
import time
import unicodedata

_STAGING = "https://prlrxjothrslhswpkoza.supabase.co"
_MIB = 1024 * 1024
# Teto LOCAL do importador Overture ajustado para 350 MiB (não aumenta a cota Supabase Free).
# Supabase Free: 500 MB (aprox. 476,8 MiB) de quota de tamanho de banco por projeto.
# Margem preventiva de 10 MiB para um lote final e expansão de índices.
_TETO_LIMITE_MIB = 350
_TETO_PREVENTIVO_MIB = _TETO_LIMITE_MIB - 10
_TETO_IMPORTADOR_MIB = _TETO_LIMITE_MIB
_PAGINA = 1000

def _normalizar(v):
    s = "".join(c for c in unicodedata.normalize("NFD", v.casefold())
                if unicodedata.category(c) != "Mn")
    return " ".join(re.sub(r"[^a-z0-9]+", " ", s).split())

def _rest(chave, rota, metodo="GET", corpo=None, contar=False):
    # JWT service_role legado usa Bearer; chave sb_secret_ nova usa só apikey.
    headers = {
        "apikey": chave,
        "Accept": "application/json", "Content-Type": "application/json",
        "Prefer": "count=exact",
    }
    if chave.startswith("eyJ"):
        headers["Authorization"] = "Bearer " + chave
    req = Request(_STAGING + rota, headers=headers, method=metodo,
                  data=json.dumps(corpo).encode("utf-8") if corpo is not None else None)
    with urlopen(req, timeout=45) as resp:
        if contar:
            total = (resp.headers.get("Content-Range") or "").rsplit("/", 1)[-1]
            if not total.isdigit():
                raise RuntimeError("O Supabase não retornou uma contagem verificável.")
            return int(total)
        return json.loads(resp.read())

def _espaco(chave):
    n = _rest(chave, "/rest/v1/rpc/overture_storage_bytes", "POST", {})
    if not isinstance(n, int) or n <= 0:
        raise RuntimeError("Falha ao consultar armazenamento da homologação.")
    return n

def _existentes(chave):
    """Paginação completa: não perde cidades após as primeiras 1000."""
    cidades = set()
    posicao = 0
    while True:
        rota = ("/rest/v1/overture_import_city_progress?select=state,city_key"
                + "&order=state.asc,city_key.asc"
                + "&limit=" + str(_PAGINA) + "&offset=" + str(posicao))
        registros = _rest(chave, rota)
        if not isinstance(registros, list):
            raise RuntimeError("Não foi possível ler a cobertura municipal existente.")
        for r in registros:
            if isinstance(r.get("state"), str) and isinstance(r.get("city_key"), str):
                cidades.add((r["state"], r["city_key"]))
        if len(registros) < _PAGINA:
            return cidades
        posicao += len(registros)

def _municipios_malha(uf, cache):
    """Depois das cidades prioritárias, reutiliza os mesmos ZIPs do IBGE do importador.

    NÃO consulta servicodados.ibge.gov.br: esse endpoint causou TimeoutError
    antes de qualquer empresa ser importada. A malha municipal de 2024 também
    é utilizada pelo importador e fica em cache em data/ibge no Colab.
    """
    if uf in cache:
        return cache[uf]
    if "_helpers" not in cache:
        from runpy import run_path
        cache["_helpers"] = run_path(str(_SCRIPT))
    helpers = cache["_helpers"]
    boundaries = helpers["ensure_state_boundaries"](uf)
    municipios = helpers["municipalities_index"](boundaries)
    nomes = {
        _normalizar(str(row.NM_MUN)): str(row.NM_MUN)
        for row in municipios.itertuples()
        if str(row.SIGLA_UF) == uf and str(row.NM_MUN).strip()
    }
    if not nomes:
        raise RuntimeError("A malha municipal do IBGE não contém municípios válidos em " + uf)
    cache[uf] = nomes
    return nomes

def _contagem(chave, uf, cidade):
    query = urlencode({
        "select": "id", "state": "eq." + uf,
        "city_key": "eq." + _normalizar(cidade),
    })
    return _rest(chave, "/rest/v1/overture_places?" + query, metodo="HEAD", contar=True)

def _painel_nacional(existentes, qtd_empresas=None, titulo="PROGRESSO NACIONAL"):
    """Sem animação ou barra repetida; imprime somente a contagem."""
    total = len(existentes)
    faltam = max(0, TOTAL_LOCALIDADES_BRASIL - total)
    print(f"{titulo}: {total:,}/{TOTAL_LOCALIDADES_BRASIL:,} municípios "
          f"({total*100/TOTAL_LOCALIDADES_BRASIL:.2f}%) | FALTAM {faltam:,}".replace(",", "."),
          flush=True)
    if qtd_empresas is not None:
        print("Empresas no banco:", f"{qtd_empresas:,}".replace(",", "."))
    return faltam

def _total_empresas(chave):
    return _rest(chave, "/rest/v1/overture_places?select=id",
                 metodo="HEAD", contar=True)

def _carregar_pendencias(chave):
    """Consulta as cidades com falhas temporariamente em espera na homologação."""
    from datetime import datetime, timezone
    agora = datetime.now(timezone.utc).isoformat(timespec="seconds")
    pendentes, offset = set(), 0
    while True:
        qs = urlencode({
            "select": "state,city_key", "retry_after": "gt." + agora,
            "order": "state.asc,city_key.asc",
            "limit": str(_PAGINA), "offset": str(offset),
        })
        itens = _rest(chave, "/rest/v1/overture_auto_city_attempts?" + qs)
        if not isinstance(itens, list):
            raise RuntimeError("Não foi possível consultar pendências.")
        pendentes.update((x["state"], x["city_key"]) for x in itens)
        if len(itens) < _PAGINA:
            return pendentes
        offset += len(itens)

def _registrar_pendencia(chave, uf, cidade, motivo):
    """Persiste adiamento sem armazenar logs, segredos ou dados confidenciais."""
    from datetime import datetime, timedelta, timezone
    prazo = timedelta(days=DIAS_ESPERA_SEM_DADOS) if motivo == "sem_dados_overture" else timedelta(hours=HORAS_ESPERA_FALHAS)
    body = {
        "state": uf, "city": cidade, "city_key": _normalizar(cidade),
        "last_error": motivo[:100],
        "retry_after": (datetime.now(timezone.utc)+prazo).isoformat(),
    }
    headers = {"apikey": chave, "Content-Type": "application/json",
               "Prefer": "resolution=merge-duplicates,return=minimal"}
    if chave.startswith("eyJ"):
        headers["Authorization"] = "Bearer " + chave
    req = Request(_STAGING + "/rest/v1/overture_auto_city_attempts?on_conflict=state,city_key",
                  method="POST", headers=headers, data=json.dumps(body).encode("utf-8"))
    with urlopen(req, timeout=40) as resp:
        resp.read()

def _classificar_falha(saida, codigo):
    texto = (saida or "").lower()
    if "no usable places" in texto or "no usable" in texto:
        return "sem_dados_overture"
    if "database reached" in texto or "database safety" in texto:
        return "teto_do_banco"
    if "unauthorized" in texto or "invalid api key" in texto:
        return "autenticacao"
    # O CLI do Overture acessa dados na nuvem; classificar o tipo de falha
    # sem imprimir o log bruto, que pode conter URLs e outros dados.
    if any(x in texto for x in ("429", "too many requests", "rate limit", "throttl")):
        return "download_overture_429"
    if any(x in texto for x in ("403", "accessdenied", "forbidden")):
        return "download_overture_403"
    if any(x in texto for x in ("connection reset", "network_connection", "curlcode", "socket")):
        return "download_overture_rede"
    if any(x in texto for x in ("timed out", "timeout", "time out")):
        return "download_overture_timeout"
    if any(x in texto for x in ("stac", "manifest")):
        return "download_overture_catalogo"
    if any(x in texto for x in ("overturemaps", "download", "connection", "s3", "duckdb")):
        return "download_overture_generico"
    return "erro_importador_" + str(codigo)


# Célula 4 — diagnóstico oficial + recuperação pontual + auditoria final.
# Usa somente a HOMOLOGAÇÃO e o importador da célula 2, sem trabalho em massa.
from datetime import datetime, timezone
from collections import Counter

def _ler_recuperacoes(chave):
    """Lê TODAS as anotações antigas, inclusive as já elegíveis, sem apagá-las."""
    itens, offset = [], 0
    while True:
        qs = urlencode({
            "select":"state,city,city_key,last_error,retry_after",
            "order":"state.asc,city_key.asc",
            "limit":str(_PAGINA), "offset":str(offset)
        })
        batch = _rest(chave, "/rest/v1/overture_auto_city_attempts?" + qs)
        if not isinstance(batch, list):
            raise RuntimeError("Não foi possível ler a fila de recuperação.")
        itens.extend(batch)
        if len(batch) < _PAGINA:
            return itens
        offset += len(batch)

def _ler_data(value):
    if not value:
        return None
    try:
        dt = datetime.fromisoformat(str(value).replace("Z","+00:00"))
        return dt.replace(tzinfo=timezone.utc) if dt.tzinfo is None else dt
    except (ValueError,TypeError):
        return None

def _montar_malha(alvos_ufs, cache, todas):
    oficiais = {}
    indisponiveis = []
    for uf in sorted(set(alvos_ufs)):
        if uf not in UFS_BRASIL:
            print("UF de recuperação inválida; ignorada:", str(uf)[:8])
            continue
        try:
            municipios = _municipios_malha(uf, cache)
            for key, nome in municipios.items():
                oficiais[(uf, key)] = nome
            print("IBGE:", uf, "| municípios na malha:", len(municipios))
        except Exception as exc:
            indisponiveis.append(uf)
            print("IBGE INDISPONÍVEL:", uf, type(exc).__name__)
    if indisponiveis:
        print("UFs sem malha (não serão importadas até baixar a fonte oficial):", ", ".join(indisponiveis))
    return oficiais, indisponiveis

def _programa_recuperacao(chave, existentes):
    recuperacoes = _ler_recuperacoes(chave)
    por_chave = {}
    for item in recuperacoes:
        uf, key = item.get("state"), item.get("city_key")
        if isinstance(uf,str) and isinstance(key,str):
            por_chave[(uf, key)] = item
    print("Anotações de recuperação (podem incluir grafias duplicadas):",len(por_chave))
    ufs = {uf for uf,_ in por_chave if uf in UFS_BRASIL}
    if not ufs:
        print("Fila de recuperação vazia: verificando a malha nacional.")
        ufs=set(UFS_BRASIL)
    cache={}
    oficiais, indisponiveis = _montar_malha(ufs,cache, False)
    if indisponiveis:
        return None, por_chave, oficiais, None

    esperados=max(0,TOTAL_LOCALIDADES_BRASIL-len(existentes))
    faltantes={key:nome for key,nome in oficiais.items() if key not in existentes}
    extras={key for key in existentes if key[0] in ufs and key not in oficiais}
    diferenca_canonica=len(faltantes)-len(extras)
    if AMPLIAR_MALHA_NACIONAL_SE_DIVERGIR and diferenca_canonica != esperados:
        print("DIVERGÊNCIA DETECTADA: ampliação da conferência para todas as UFs.")
        restante=[uf for uf in UFS_BRASIL if uf not in ufs]
        ampliado, erros=_montar_malha(restante,cache,True)
        if erros:
            return None, por_chave, oficiais, None
        oficiais.update(ampliado)
        faltantes={key:nome for key,nome in oficiais.items() if key not in existentes}
        extras={key for key in existentes if key not in oficiais}
    print("Ausentes segundo a MALHA IBGE analisada:",len(faltantes))
    print("Registros de progresso sem correspondência exata nessa malha:",len(extras))
    print("Diferença numérica nacional:",esperados,
          "| diferença canônica no escopo:",len(faltantes)-len(extras))
    print("As listas podem divergir por nomes alternativos; nunca exclua registros para ajustar a conta.")
    if extras:
        print("GRAFIAS PARA REVISAR:",", ".join(f"{k[0]}/{k[1]}" for k in sorted(extras)[:12]))
        print("A diferença numérica de 18 não substitui a conferência oficial.")
    for uf,key in sorted(faltantes):
        reg=por_chave.get((uf,key),{})
        prazo=_ler_data(reg.get("retry_after"))
        situacao="AGUARDAR" if prazo and prazo > datetime.now(timezone.utc) else "LIBERADA"
        detalhe=(reg.get("last_error") or "sem erro anterior")
        print(f"ALVO {uf}/{faltantes[(uf,key)]}: {situacao} | {detalhe}" +
              (f" | após {prazo.isoformat()}" if situacao=="AGUARDAR" else ""))
    nao_oficiais=[(uf,key) for uf,key in por_chave
                  if (uf,key) not in oficiais and (uf,key) not in existentes]
    if nao_oficiais:
        print("Anotações antigas não presentes na malha: NÃO serão forçadas:",
              ", ".join(f"{uf}/{key}" for uf,key in sorted(nao_oficiais)[:20]))
    return faltantes,por_chave,oficiais,extras

def _executar_importador_final(chave,uf,cidade,deadline):
    cmd=[sys.executable,str(_SCRIPT),
         "--states",uf,"--cities",cidade,
         "--max-records-per-state",str(LIMITE_POR_MUNICIPIO),
         "--max-records-per-city",str(LIMITE_POR_MUNICIPIO),
         "--database-max-mb",str(_TETO_IMPORTADOR_MIB)]
    env=dict(os.environ)
    env["SUPABASE_URL"]=_STAGING
    env["SUPABASE_SERVICE_ROLE_KEY"]=chave
    motivo=None
    try:
        for tentativa in range(2):
            restante=deadline-time.monotonic()-90
            if restante < 90:
                return "fim_do_tempo"
            try:
                resultado=subprocess.run(cmd,env=env,check=False,
                    stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                    text=True,errors="replace",timeout=min(1200,int(restante)))
                if resultado.returncode==0:
                    return None
                motivo=_classificar_falha(resultado.stdout,resultado.returncode)
            except subprocess.TimeoutExpired:
                motivo="timeout_importacao"
            if tentativa==0 and (motivo=="timeout_importacao" or
                                 (motivo or "").startswith("download_overture")):
                print("REPETINDO DOWNLOAD:",cidade+"/"+uf,"(tentativa 2)")
                time.sleep(ESPERA_ENTRE_TENTATIVAS)
                continue
            return motivo
    finally:
        env.pop("SUPABASE_SERVICE_ROLE_KEY",None)
        del env
    return motivo

def _resumo_final(chave, inicio, lista_confirmados, falhas, faltantes, oficiais):
    existentes_finais=_existentes(chave)
    empresas=_total_empresas(chave)
    numeric_missing=max(0,TOTAL_LOCALIDADES_BRASIL-len(existentes_finais))
    ainda_oficiais=[(uf,key,nome) for (uf,key),nome in faltantes.items()
                    if (uf,key) not in existentes_finais]
    print("\n"+"="*58)
    print("RESULTADO FINAL — FASE 7 (conferido no Supabase)")
    print("Total marcado no progresso:",len(existentes_finais),"/",TOTAL_LOCALIDADES_BRASIL)
    print("Diferença numérica:",numeric_missing)
    print("Empresas no banco:",empresas)
    print("Novos municípios desta execução:",len(lista_confirmados))
    print("Falhas nesta execução:",len(falhas))
    print("Ausentes entre os municípios-alvo da malha:",len(ainda_oficiais))
    print("Armazenamento:",round(_espaco(chave)/_MIB,2),"MiB")
    if ainda_oficiais:
        for uf,key,nome in ainda_oficiais:
            print("AINDA FALTA:",nome+"/"+uf)
    if len(existentes_finais)==TOTAL_LOCALIDADES_BRASIL and not ainda_oficiais:
        if len(oficiais)>=TOTAL_LOCALIDADES_BRASIL:
            print("META NUMÉRICA ATINGIDA; todos os municípios conferidos na malha carregada.")
        else:
            print("META NUMÉRICA ATINGIDA, mas conferência IBGE foi parcial.")
            print("É necessário auditar todos os estados antes de afirmar cobertura nacional exata.")
    elif not ainda_oficiais and len(existentes_finais)>TOTAL_LOCALIDADES_BRASIL:
        print("DIVERGÊNCIA: existem mais chaves no progresso que o total nacional esperado.")
        print("Reveja aliases e grafias extras antes de declarar a meta concluída.")
    else:
        print("AINDA NÃO CONCLUÍDO. Reexecute só esta célula quando as recuperações forem liberadas.")
    print("A amostra de 60 empresas por cidade não representa todas as empresas locais.")
    print("="*58)
    return existentes_finais

def executar_recuperacao_final():
    print("Januário Leads — recuperar municípios pendentes sem reiniciar a importação.")
    print("Destino exclusivo: HOMOLOGAÇÃO | Parada preventiva:",_TETO_PREVENTIVO_MIB,"MiB")
    confirmar=input("Para iniciar somente cidades faltantes, digite RECUPERAR_18: ").strip().upper()
    if confirmar!="RECUPERAR_18":
        print("Cancelado antes de acessar o banco.")
        return
    chave=getpass("Chave SECRETA da HOMOLOGAÇÃO (oculta): ").strip()
    if not chave or chave.startswith(("sb_publishable_","sbp_")) or not chave.startswith(("sb_secret_","eyJ")):
        print("Chave inválida ou pública; nada foi executado.")
        chave=""
        return
    try:
        armazenamento=_espaco(chave)
        if armazenamento >= _TETO_PREVENTIVO_MIB*_MIB:
            print("PARADA SEGURA:",round(armazenamento/_MIB,2),"MiB no banco.")
            return
        existentes=_existentes(chave)
        _painel_nacional(existentes,qtd_empresas=_total_empresas(chave),
                         titulo="INÍCIO — CONFERIDO NO SUPABASE")
        alvos,recuperacoes,oficiais,extras=_programa_recuperacao(chave,existentes)
        if alvos is None:
            print("SEM IMPORTAÇÃO: não foi possível validar os municípios na fonte IBGE.")
            return
        if not alvos:
            print("Nenhum município ausente na malha consultada.")
            _resumo_final(chave,existentes,[],[],alvos,oficiais)
            return
        # Preserve cooldown. Avoid a "ghost" city from the old 22-item queue.
        fila=[]
        esperando=[]
        agora=datetime.now(timezone.utc)
        for (uf,key),cidade in sorted(alvos.items()):
            registro=recuperacoes.get((uf,key),{})
            data=_ler_data(registro.get("retry_after"))
            if RESPEITAR_AGENDAMENTO_RECUPERACAO and data and data>agora:
                esperando.append((uf,cidade,data))
            else:
                fila.append((uf,key,cidade))
        print("Cidades IBGE liberadas agora:",len(fila),
              "| aguardando a data de retorno:",len(esperando))
        for uf,cidade,dt in esperando:
            print("AGUARDANDO:",cidade+"/"+uf,"| nova tentativa:",dt.isoformat())
        if not fila:
            print("Nenhuma cidade liberada agora; nenhum dado alterado.")
            return
        deadline=time.monotonic()+TEMPO_MAX_MINUTOS*60
        confirmados=[];falhas=[];ufs_falha_download=set()
        falhas_consecutivas=0
        limite=min(MAX_NOVOS_POR_SESSAO,len(fila))
        for uf,key,cidade in fila[:limite]:
            if time.monotonic()>deadline-180:
                print("PARADA SEGURA: limite de tempo da sessão.")
                break
            if len(falhas)>=MAX_FALHAS_POR_EXECUCAO or falhas_consecutivas>=MAX_FALHAS_CONSECUTIVAS:
                print("PAUSA: repetição de falhas. Tente novamente mais tarde.")
                break
            if len(ufs_falha_download)>=MAX_UFS_COM_FALHA_DOWNLOAD:
                print("PAUSA DE REDE: falhas do Overture em",len(ufs_falha_download),"UFs.")
                break
            if _espaco(chave)>=_TETO_PREVENTIVO_MIB*_MIB:
                print("PAUSA: armazenamento próximo do limite preventivo.")
                break
            existentes_atual=_existentes(chave)
            if (uf,key) in existentes_atual:
                print("PULADO, já salvo:",cidade+"/"+uf)
                continue
            antes=_contagem(chave,uf,cidade)
            if antes>0:
                print("REVISAR: já há",antes,"empresas de",cidade+"/"+uf,
                      "mas não há progresso registrado; nenhum dado foi sobrescrito.")
                falhas.append((uf,cidade,"registros_sem_progresso"))
                continue
            print("IMPORTANDO:",cidade+"/"+uf,"| amostra máx.",LIMITE_POR_MUNICIPIO)
            motivo=_executar_importador_final(chave,uf,cidade,deadline)
            depois=_contagem(chave,uf,cidade)
            progresso_novo=_existentes(chave)
            if depois>antes and (uf,key) in progresso_novo:
                confirmados.append((uf,cidade,depois))
                falhas_consecutivas=0
                print("SALVO:",cidade+"/"+uf,"| empresas:",depois,
                      "| municípios:",len(progresso_novo),"/",TOTAL_LOCALIDADES_BRASIL)
            else:
                motivo=motivo or "sem_registros_confirmados"
                if motivo in ("autenticacao","teto_do_banco","fim_do_tempo"):
                    print("PARADA:",motivo,"|",cidade+"/"+uf)
                    falhas.append((uf,cidade,motivo))
                    break
                falhas.append((uf,cidade,motivo))
                falhas_consecutivas+=1
                if motivo.startswith("download_overture") or motivo=="timeout_importacao":
                    ufs_falha_download.add(uf)
                try:
                    _registrar_pendencia(chave,uf,cidade,motivo)
                    print("PENDENTE:",cidade+"/"+uf,"| motivo:",motivo,
                          "| reavaliar após",("7 dias" if motivo=="sem_dados_overture" else "6 horas"))
                except Exception as exc:
                    print("PENDENTE:",cidade+"/"+uf,"| falha ao salvar agendamento:",type(exc).__name__)
            time.sleep(ESPERA_ENTRE_CIDADES)
        _resumo_final(chave,existentes,confirmados,falhas,alvos,oficiais)
        print("PRÓXIMO PASSO: envie o RESULTADO FINAL ao ChatGPT. Não compartilhe a chave.")
    except HTTPError as exc:
        if exc.code in (401,403):
            print("Acesso negado pelo Supabase. Confira a chave SECRETA da HOMOLOGAÇÃO.")
        else:
            print("Consulta HTTP falhou (código):",exc.code)
    except Exception as exc:
        print("Importação interrompida com segurança. Tipo de falha:",type(exc).__name__)
        print("Os municípios já gravados permanecem no Supabase.")
        raise
    finally:
        chave=""
        print("Credencial encerrada. Desconecte o runtime ao terminar.")

executar_recuperacao_final()


## Como interpretar o resultado

- **SALVO**: há empresas **e** uma linha de progresso confirmada no Supabase.
- **AGUARDANDO**: ainda está em período de recuperação; não forçar.
- **REVISAR**: existem empresas sem registro correspondente de progresso; exige investigação para evitar inconsistências.
- **PENDENTE**: a fonte não respondeu ou não retornou empresas utilizáveis; o Colab agendou uma nova tentativa.
- **DIVERGÊNCIA**: o número de registros não corresponde exatamente ao catálogo municipal IBGE; o notebook mostra as grafias para revisão, sem declarar conclusão indevida.

Quando finalizar, copie apenas o resumo e os motivos de erro. **Nunca copie a chave secreta.**